In [ ]:
!pip install -q faster-whisper gradio yt-dlp pydub ffmpeg-python nltk transformers sentence-transformers faiss-cpu deep-translator google-generativeai openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 47.8 MB/s eta 0:00:00


In [ ]:
import os
import yt_dlp
import gradio as gr
import numpy as np
import torch
import nltk
import time
from pydub import AudioSegment
from faster_whisper import WhisperModel
from nltk.tokenize import sent_tokenize
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
import faiss
from deep_translator import GoogleTranslator

# Auto-detect hardware
device = "cuda" if torch.cuda.is_available() else "cpu"
compute = "float16" if device == "cuda" else "int8"
print(f"၀ Running on: {device.upper()}")

nltk.download('punkt')
nltk.download('punkt_tab')

# Global config
languages = {"Tamil": "ta", "Hindi": "hi", "Telugu": "te", "Malayalam": "ml", "English": "en"}

၀ Running on: CPU


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
from openai import OpenAI

# Initialize Groq Client
groq_api_key = "paste here your api key"
groq_client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

print("✅ Groq client (Fast Mode) initialized successfully.")

✅ Groq client (Fast Mode) initialized successfully.


In [ ]:
def download_video(url):
    ydl_opts = {'format': 'bestaudio/best', 'outtmpl': 'input.%(ext)s'}
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])
    return "input.webm"

In [ ]:
import subprocess

def convert_to_wav(file):
    # Faster conversion using ffmpeg directly
    output = "audio.wav"
    subprocess.run(['ffmpeg', '-y', '-i', file, '-ar', '16000', '-ac', '1', 'audio.wav'], check=True)
    return output

def split_audio(file, chunk_length_sec=600): # 10 minute chunks for less overhead
    chunks = []
    # Use ffmpeg for high-speed segmenting
    subprocess.run([
        'ffmpeg', '-y', '-i', file,
        '-f', 'segment',
        '-segment_time', str(chunk_length_sec),
        '-c', 'copy', 'out_chunk_%03d.wav'
    ], check=True)

    import glob
    chunks = sorted(glob.glob("out_chunk_*.wav"))
    return chunks

In [ ]:
import re

def clean_text(text):
    if not text:
        return ""

    # Split into sentences
    sentences = sent_tokenize(text)
    clean = []

    for s in sentences:
        s = s.strip()
        # Filter out very short segments
        if len(s) < 10: continue

        # Filter out repetitive noise like 'N-N-N' or '...-...' using regex
        # This checks for a character followed by a hyphen repeated 3 or more times
        if re.search(r'(\w-){3,}', s) or re.search(r'([A-Za-z])\1{4,}', s):
            continue

        clean.append(s)

    return " ".join(clean)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Initialize Summarization Model (BART)
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
summarizer_model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

def summarize_text(text):
    if not text or len(text.strip().split()) < 20:
        return "Text too short to provide a meaningful summary."

    try:
        words = text.split()
        # Chunking text for BART processing
        chunk_size = 700
        chunks = [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]

        summaries = []
        for chunk in chunks:
            inputs = tokenizer([chunk], max_length=1024, return_tensors="pt", truncation=True).to(device)
            summary_ids = summarizer_model.generate(
                inputs["input_ids"],
                num_beams=4,
                max_length=300,
                min_length=80,
                repetition_penalty=2.0,
                length_penalty=1.2,
                early_stopping=True
            )
            decoded = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
            summaries.append(decoded)

        # Combine and format into clean bullet points
        combined = " ".join(summaries)
        final_sentences = sent_tokenize(combined)
        bullet_summary = "\n".join([f"• {s}" for s in final_sentences if len(s) > 20])

        return bullet_summary
    except Exception as e:
        return f"Summarization Error: {str(e)}"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [ ]:
import time
from faster_whisper import WhisperModel

# Using 'tiny' model for maximum speed on CPU with faster-whisper
model = WhisperModel("tiny", device=device, compute_type=compute)

def transcribe_chunks(chunks):
    full_text = ""
    if not chunks:
        return "No audio chunks found."

    print(f"⚡ Fast Transcribing {len(chunks)} large chunks using faster-whisper...")
    start_trans = time.time()

    for i, chunk in enumerate(chunks):
        try:
            # faster-whisper returns a generator for segments
            segments, info = model.transcribe(chunk, beam_size=1, vad_filter=True)
            for seg in segments:
                full_text += seg.text + " "
        except Exception as e:
            print(f"❌ Error in chunk {i}: {e}")

    end_trans = time.time()
    print(f"✅ Completed in {end_trans - start_trans:.2f}s")
    return full_text.strip()

In [ ]:
languages = {"Tamil": "ta", "Hindi": "hi", "Telugu": "te", "Malayalam": "ml", "English": "en"}

def translate_text(text, lang):
    if not text or text.strip() == "":
        return ""
    try:
        target_code = languages.get(lang, "en")
        if target_code == "en":
            return text
        translator = GoogleTranslator(source='auto', target=target_code)
        max_chars = 3000
        if len(text) > max_chars:
            parts = [text[i:i+max_chars] for i in range(0, len(text), max_chars)]
            translated_parts = [translator.translate(p) for p in parts if p.strip()]
            valid_parts = [p for p in translated_parts if p]
            return " ".join(valid_parts)
        result = translator.translate(text)
        return result if result else "Translation returned empty result."
    except Exception as e:
        return f"Translation Error: {str(e)}"

In [ ]:
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# Initialize global variables to prevent NameError
faiss_index = None
stored_sentences = []

def create_faiss_index(text):
    sentences = sent_tokenize(text)
    if not sentences:
        return None, []
    embeddings = embed_model.encode(sentences)

    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(np.array(embeddings))

    return index, sentences

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def answer_question(question, index=None, sentences=None):
    """Direct Q&A using Groq Llama 3.3 for high speed."""
    try:
        # Construct context if available
        context_text = ""
        if sentences and len(sentences) > 0:
            context_text = "Context from lecture: " + " ".join(sentences[:50])
            system_instruction = f"You are an AI Lecture Assistant. Use the following context to answer: {context_text}"
        else:
            system_instruction = "You are a helpful AI Assistant. Answer the user's question directly."

        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": question}
            ],
            max_tokens=500,
            temperature=0.7
        )

        return response.choices[0].message.content
    except Exception as e:
        return f"Error contacting Groq: {str(e)}"

In [ ]:
import time

test_prompt = "Say hello in one word."
print(f'Testing Groq API Latency (Llama 3.3 70b)...')

try:
    start = time.time()
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": test_prompt}],
        max_tokens=10
    )
    end = time.time()
    print(f'\nResponse: {response.choices[0].message.content.strip()}')
    print(f'Latency: {end - start:.4f} seconds')
except Exception as e:
    print(f'\nError during speed test: {e}')

Testing Groq API Latency (Llama 3.3 70b)...

Response: Hello.
Latency: 1.3410 seconds


In [ ]:
cancel_processing = False

def process(input_file, link, language):
    global cancel_processing
    cancel_processing = False
    try:
        if link and link.strip() != "":
            video = download_video(link)
            audio = convert_to_wav(video)
        elif input_file is not None:
            audio = convert_to_wav(input_file.name)
        else:
            return "No input provided", "", "", "", "", "", "", ""

        if cancel_processing: return "Cancelled", "", "", "", "", "", "", ""

        chunks = split_audio(audio)
        if not chunks: return "Chunking failed", "", "", "", "", "", "", ""

        # Process chunks with cancellation check
        full_transcript = ""
        print(f"⚡ Fast Transcribing {len(chunks)} chunks...")
        for i, chunk in enumerate(chunks):
            if cancel_processing:
                return "Processing Cancelled", "", "", "", "", "", "", ""
            segments, info = model.transcribe(chunk, beam_size=1, vad_filter=True)
            for seg in segments:
                full_transcript += seg.text + " "

        full_clean = clean_text(full_transcript)
        full_summary = summarize_text(full_clean)
        full_translated_transcript = translate_text(full_clean, language)
        full_translated_summary = translate_text(full_summary, language)

        global faiss_index, stored_sentences
        faiss_index, stored_sentences = create_faiss_index(full_clean)

        def ui_preview(text, limit=150):
            words = text.split()
            if len(words) > limit:
                return " ".join(words[:limit]) + "... [Full text available in Download]"
            return text

        return (
            ui_preview(full_clean), ui_preview(full_summary),
            ui_preview(full_translated_transcript), ui_preview(full_translated_summary),
            full_clean, full_summary,
            full_translated_transcript, full_translated_summary
        )

    except Exception as e:
        return f"Error: {str(e)}", "Error", "Error", "Error", "", "", "", ""

In [ ]:
def chat(question):
    return answer_question(question, faiss_index, stored_sentences)

In [ ]:
def save_file(text, filename):
    with open(filename, "w") as f:
        f.write(text)
    return filename

In [ ]:
import gradio as gr
import os
import time

# ---------------- SAVE FILE ----------------
def save_and_get_path(text, label):
    if not text or text.strip() == "":
        return None
    filename = f"{label.lower().replace(' ', '_')}.txt"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(text)
    return filename

# ---------------- VISIBILITY CONTROL ----------------
def show_text_mode():
    return (
        gr.update(visible=True),   # text_section
        gr.update(visible=True),   # summary_section
        gr.update(visible=False),  # tclean_section
        gr.update(visible=False)   # tsummary_section
    )

def show_translation_mode():
    return (
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=True),
        gr.update(visible=True)
    )

with gr.Blocks() as demo:
    gr.Markdown("# Ħ AI Lecture Analysis ")

    full_clean_state = gr.State("")
    full_summary_state = gr.State("")
    full_t_clean_state = gr.State("")
    full_t_summary_state = gr.State("")

    # ---------------- INPUT ----------------
    with gr.Row():
        with gr.Column():
            file_input = gr.File(label="Upload Audio/Video")
            link_input = gr.Textbox(label="Or Paste Video Link")
            lang = gr.Dropdown(choices=list(languages.keys()), value="English", label="Target Language (for Files)")
            process_all_btn = gr.Button("⚙‸ Process All", variant="primary")

    # ---------------- MODE BUTTONS ----------------
    with gr.Row():
        btn_text_mode = gr.Button("📝 Text")
        btn_translation_mode = gr.Button("🌍 Translation")

    # ---------------- TEXT SECTIONS ----------------
    with gr.Row():

        with gr.Column(visible=True) as text_section:
            gr.Markdown("### 1. Cleaned Text")
            clean_out = gr.Textbox(label="Preview", lines=5)
            with gr.Row():
                btn_gen_clean = gr.Button("Generate Cleaned")
                btn_dl_clean = gr.Button("⌄ Download Full")
            file_dl_clean = gr.File(label="Download Link", visible=False)

        with gr.Column(visible=True) as summary_section:
            gr.Markdown("### 2. Summary (Points)")
            summary_out = gr.Textbox(label="Preview", lines=5)
            with gr.Row():
                btn_gen_sm = gr.Button("Generate Summary")
                btn_dl_sm = gr.Button("⌄ Download Full")
            file_dl_sm = gr.File(label="Download Link", visible=False)

    # ---------------- TRANSLATION SECTIONS ----------------
    with gr.Row():

        with gr.Column(visible=False) as tclean_section:
            gr.Markdown("### 3. Translated Cleaned")
            t_clean_out = gr.Textbox(label="Preview", lines=5)
            with gr.Row():
                btn_gen_tcl = gr.Button("Generate Translation")
                btn_dl_tcl = gr.Button("⌄ Download Full")
            file_dl_tcl = gr.File(label="Download Link", visible=False)

        with gr.Column(visible=False) as tsummary_section:
            gr.Markdown("### 4. Translated Summary")
            t_summary_out = gr.Textbox(label="Preview", lines=5)
            with gr.Row():
                btn_gen_tsm = gr.Button("Generate Trans. Summary")
                btn_dl_tsm = gr.Button("⌄ Download Full")
            file_dl_tsm = gr.File(label="Download Link", visible=False)

    # ---------------- CHATBOT ----------------
    gr.Markdown("## Ħ Context-Aware Chatbot")
    chat_lang = gr.Dropdown(choices=list(languages.keys()), value="English", label="Chat Assistant Language")
    chatbot = gr.Chatbot(label="Lecture Assistant", type='messages')
    msg = gr.Textbox(placeholder="Ask a question about the lecture...")
    clear = gr.Button("Clear Chat")

    def respond(message, chat_history, selected_lang):
        if not message:
            return "", chat_history

        chat_history.append({"role": "user", "content": message})
        reply = answer_question(message, faiss_index, stored_sentences)

        if selected_lang != "English":
            reply = translate_text(reply, selected_lang)

        chat_history.append({"role": "assistant", "content": reply})
        return "", chat_history

    def run_process(f, l, ln):
        return process(f, l, ln)

    # ---------------- PROCESS ----------------
    process_all_btn.click(
        run_process,
        [file_input, link_input, lang],
        [clean_out, summary_out, t_clean_out, t_summary_out,
         full_clean_state, full_summary_state,
         full_t_clean_state, full_t_summary_state]
    )

    # ---------------- MODE SWITCH ----------------
    btn_text_mode.click(
        show_text_mode,
        [],
        [text_section, summary_section, tclean_section, tsummary_section]
    )

    btn_translation_mode.click(
        show_translation_mode,
        [],
        [text_section, summary_section, tclean_section, tsummary_section]
    )

    # ---------------- GENERATE ----------------
    btn_gen_clean.click(lambda f, l, ln: run_process(f, l, ln)[0], [file_input, link_input, lang], clean_out)
    btn_gen_sm.click(lambda f, l, ln: run_process(f, l, ln)[1], [file_input, link_input, lang], summary_out)
    btn_gen_tcl.click(lambda f, l, ln: run_process(f, l, ln)[2], [file_input, link_input, lang], t_clean_out)
    btn_gen_tsm.click(lambda f, l, ln: run_process(f, l, ln)[3], [file_input, link_input, lang], t_summary_out)

    # ---------------- DOWNLOAD ----------------
    btn_dl_clean.click(lambda x: save_and_get_path(x, "cleaned"), full_clean_state, file_dl_clean).then(lambda: gr.update(visible=True), None, file_dl_clean)
    btn_dl_sm.click(lambda x: save_and_get_path(x, "summary"), full_summary_state, file_dl_sm).then(lambda: gr.update(visible=True), None, file_dl_sm)
    btn_dl_tcl.click(lambda x: save_and_get_path(x, "translated_clean"), full_t_clean_state, file_dl_tcl).then(lambda: gr.update(visible=True), None, file_dl_tcl)
    btn_dl_tsm.click(lambda x: save_and_get_path(x, "translated_summary"), full_t_summary_state, file_dl_tsm).then(lambda: gr.update(visible=True), None, file_dl_tsm)

    msg.submit(respond, [msg, chatbot, chat_lang], [msg, chatbot])
    clear.click(lambda: [], None, chatbot, queue=False)

demo.launch(share=True)

/tmp/ipykernel_2345/1227203204.py:93: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Lecture Assistant", type='messages')


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6f4e3af6077f47a907.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
